In [1]:
import os, pickle, glob
from pyspark.sql import SparkSession
from pyspark.ml.feature import Word2Vec

In [2]:
spark = SparkSession.builder \
    .config("spark.driver.memory", "32g") \
	.config("spark.executor.memory", "128g") \
    .config('spark.executor.instances', 31) \
	.appName("Word2Vec") \
	.getOrCreate()

In [3]:
#df = spark.read.csv("shared/all_reviews.csv", header=True, inferSchema=True)
df = spark.read.csv(
    "shared/all_reviews.csv",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    multiLine=True
)

In [4]:
df

DataFrame[recommendationid: int, appid: int, game: string, author_steamid: bigint, author_num_games_owned: int, author_num_reviews: int, author_playtime_forever: int, author_playtime_last_two_weeks: int, author_playtime_at_review: int, author_last_played: int, language: string, review: string, timestamp_created: int, timestamp_updated: int, voted_up: int, votes_up: int, votes_funny: bigint, weighted_vote_score: double, comment_count: bigint, steam_purchase: int, received_for_free: int, written_during_early_access: int, hidden_in_steam_china: int, steam_china_location: string]

In [6]:
df.columns

['recommendationid',
 'appid',
 'game',
 'author_steamid',
 'author_num_games_owned',
 'author_num_reviews',
 'author_playtime_forever',
 'author_playtime_last_two_weeks',
 'author_playtime_at_review',
 'author_last_played',
 'language',
 'review',
 'timestamp_created',
 'timestamp_updated',
 'voted_up',
 'votes_up',
 'votes_funny',
 'weighted_vote_score',
 'comment_count',
 'steam_purchase',
 'received_for_free',
 'written_during_early_access',
 'hidden_in_steam_china',
 'steam_china_location']

# Model Selection

In [20]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer,
    StringIndexer,
    OneHotEncoder,
    VectorAssembler,
    StandardScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [21]:
sample_df = df.sample(False, 0.01, seed=42)

In [22]:
# categorizing columns
numeric_cols = [
    "author_num_games_owned",
    "author_num_reviews",
    "author_playtime_forever",
    "author_playtime_last_two_weeks",
    "author_playtime_at_review",
    "votes_up",
    "votes_funny",
    "weighted_vote_score",
    "comment_count"
]

categorical_cols = [
    "game",
    "language",
    "steam_china_location"
]

boolean_cols = [
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "hidden_in_steam_china"
]

label_col = "voted_up"

In [23]:
# casting numeric columns
for c in numeric_cols:
    sample_df = sample_df.withColumn(c, F.col(c).cast("double"))

In [24]:
# casting boolean columns
for c in boolean_cols:
    sample_df = sample_df.withColumn(c, F.col(c).cast("double"))

# cast label to double
sample_df = sample_df.withColumn(label_col, F.col(label_col).cast("double"))

# remove rows with null labels
sample_df = sample_df.filter(F.col(label_col).isNotNull())

In [25]:
# fill null boolean columns
for c in boolean_cols:
    sample_df = sample_df.withColumn(
        c,
        F.when(F.col(c).isNull() | F.isnan(F.col(c)), 0.0)
         .otherwise(F.col(c))
    )

In [26]:
# feature engineering
sample_df = sample_df.withColumn(
    "review_length",
    F.coalesce(F.length(F.col("review")), F.lit(0)).cast("double")
)

sample_df = sample_df.withColumn(
    "playtime_hours",
    (
        F.coalesce(F.col("author_playtime_forever"), F.lit(0.0)) / F.lit(60.0)
    ).cast("double")
)

sample_df = sample_df.withColumn(
    "helpful_ratio",
    (
        F.coalesce(F.col("votes_up"), F.lit(0.0)) /
        (F.coalesce(F.col("votes_funny"), F.lit(0.0)) + F.lit(1.0))
    ).cast("double")
)

In [27]:
# clean null and NaNs with numeric values
pre_pipeline_numeric_cols = numeric_cols + [
    "review_length",
    "playtime_hours",
    "helpful_ratio"
]

for c in pre_pipeline_numeric_cols:
    sample_df = sample_df.withColumn(
        c,
        F.when(F.col(c).isNull() | F.isnan(F.col(c)), 0.0)
         .otherwise(F.col(c))
    )

In [28]:
# fill missing categorical values
for c in categorical_cols:
    sample_df = sample_df.withColumn(
        c,
        F.when(F.col(c).isNull(), "Unknown")
         .otherwise(F.col(c).cast("string"))
    )

In [29]:
# train test split
train_df, test_df = sample_df.randomSplit([0.8, 0.2], seed=42)

In [30]:
# impute numeric columns
imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=[f"{c}_imputed" for c in numeric_cols]
).setStrategy("median")

In [31]:
# encode categorical columns
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f"{c}_index",
        handleInvalid="keep"
    )
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_index" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

In [32]:
# assemble and scale numeric features
numeric_feature_cols = [
    "review_length",
    "playtime_hours",
    "helpful_ratio"
] + [f"{c}_imputed" for c in numeric_cols]

numeric_assembler = VectorAssembler(
    inputCols=numeric_feature_cols,
    outputCol="numeric_features",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="numeric_features",
    outputCol="scaled_numeric_features",
    withMean=False,
    withStd=True
)

In [33]:
# final feature vector
final_feature_cols = (
    ["scaled_numeric_features"] +
    [f"{c}_ohe" for c in categorical_cols] +
    boolean_cols
)

final_assembler = VectorAssembler(
    inputCols=final_feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

In [34]:
# random forest classifier
rf = RandomForestClassifier(
    labelCol="voted_up",
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=50,
    maxDepth=10,
    seed=42
)

In [35]:
# model pipeline
model_pipeline = Pipeline(stages=
    [imputer] +
    indexers +
    [
        encoder,
        numeric_assembler,
        scaler,
        final_assembler,
        rf
    ]
)

In [36]:
# target values checking
train_df.select("voted_up").distinct().show()

+--------+
|voted_up|
+--------+
|     0.0|
|     1.0|
+--------+



In [37]:
# fit model
model = model_pipeline.fit(train_df)

In [38]:
# predictions
train_predictions = model.transform(train_df)
test_predictions = model.transform(test_df)

In [39]:
# model evaluation
auc_evaluator = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
)

train_auc = auc_evaluator.evaluate(train_predictions)
test_auc = auc_evaluator.evaluate(test_predictions)

train_accuracy = accuracy_evaluator.evaluate(train_predictions)
test_accuracy = accuracy_evaluator.evaluate(test_predictions)

print(f"Train AUC: {train_auc}")
print(f"Test AUC: {test_auc}")
print(f"Train Accuracy: {train_accuracy}")
print(f"Test Accuracy: {test_accuracy}")

Train AUC: 0.7657615900860155
Test AUC: 0.7624049083243485
Train Accuracy: 0.8560705781351379
Test Accuracy: 0.8562484590187031


In [40]:
# example prediction
test_predictions.select(
    label_col,
    "prediction",
    "probability"
).show(20, truncate=False)

+--------+----------+----------------------------------------+
|voted_up|prediction|probability                             |
+--------+----------+----------------------------------------+
|1.0     |1.0       |[0.1505877174566736,0.8494122825433265] |
|1.0     |1.0       |[0.1640905550759023,0.8359094449240977] |
|1.0     |1.0       |[0.16639221444884816,0.8336077855511519]|
|1.0     |1.0       |[0.14254326422475325,0.8574567357752467]|
|1.0     |1.0       |[0.13513118609025893,0.864868813909741] |
|1.0     |1.0       |[0.13279211611612238,0.8672078838838776]|
|1.0     |1.0       |[0.17052658625112335,0.8294734137488767]|
|1.0     |1.0       |[0.14443629544361813,0.8555637045563819]|
|1.0     |1.0       |[0.1372212981315865,0.8627787018684134] |
|1.0     |1.0       |[0.13513118609025893,0.864868813909741] |
|1.0     |1.0       |[0.12870714213097217,0.8712928578690278]|
|0.0     |1.0       |[0.17559845197601065,0.8244015480239894]|
|1.0     |1.0       |[0.14516082183023854,0.85483917816

## Model 2 Selection

In [ ]:
# Random Forest model
rf_model_2 = RandomForestClassifier(
    labelCol=label_col,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=15,
    seed=42
)

# Pipeline
model_pipeline_2 = Pipeline(stages=
    [imputer] +
    indexers +
    [
        encoder,
        numeric_assembler,
        scaler,
        final_assembler,
        rf_model_2
    ]
)

# Train model
model_2 = model_pipeline_2.fit(train_df)

# Predictions
train_predictions_2 = model_2.transform(train_df)
test_predictions_2 = model_2.transform(test_df)

auc_evaluator = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
)

# Metrics
print("Model 2 Train AUC:", auc_evaluator.evaluate(train_predictions_2))
print("Model 2 Test AUC:", auc_evaluator.evaluate(test_predictions_2))

print("Model 2 Train Accuracy:", accuracy_evaluator.evaluate(train_predictions_2))
print("Model 2 Test Accuracy:", accuracy_evaluator.evaluate(test_predictions_2))

In [ ]:
# model 2 - random forest classifier
rf_model_2 = RandomForestClassifier(
    labelCol=label_col,
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=100,
    maxDepth=15,
    seed=42
)

In [ ]:
# model 2 - pipeline
model_pipeline_2 = Pipeline(stages=
    [imputer] +
    indexers +
    [
        encoder,
        numeric_assembler,
        scaler,
        final_assembler,
        rf_model_2
    ]
)


In [ ]:
# model 2 - train model
model_2 = model_pipeline_2.fit(train_df)

In [ ]:
# model 2 - predictions
train_predictions_2 = model_2.transform(train_df)
test_predictions_2 = model_2.transform(test_df)

In [ ]:
# model 2 - metrics
auc_evaluator = BinaryClassificationEvaluator(
    labelCol=label_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol=label_col,
    predictionCol="prediction",
    metricName="accuracy"
)

print("Model 2 Train AUC:", auc_evaluator.evaluate(train_predictions_2))
print("Model 2 Test AUC:", auc_evaluator.evaluate(test_predictions_2))
print("Model 2 Train Accuracy:", accuracy_evaluator.evaluate(train_predictions_2))
print("Model 2 Test Accuracy:", accuracy_evaluator.evaluate(test_predictions_2))